# F1 Race Result Predictor - Model Training

Trains two models on F1 race data from the 2022–2024 seasons:
- **Random Forest Regressor** (baseline) via `RandomizedSearchCV`
- **XGBoost Ranker** (final model) via `BayesSearchCV` with a `rank:ndcg` objective

Both trained models are saved to disk with `pickle` for use in `evaluate.ipynb`.

**Data source:** Per-race CSV files organised by season folder (`2022/`, `2023/`, `2024/`).  
Each row is one driver entry; columns include qualifying position, constructor, tyre data, and historical features.  
Target column: `FinishingPos`.

## 1. Imports

In [2]:
import glob
import pickle

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from skopt import BayesSearchCV
from skopt.space import Integer, Real

## 2. Data Loading

In [3]:
def read_and_concat(file_list):
    """Read a list of CSV files and concatenate into a single DataFrame."""
    dfs = [pd.read_csv(f) for f in file_list]
    return pd.concat(dfs, ignore_index=True)

In [ ]:
file_list_2022 = glob.glob("../data/2022/*.csv")
file_list_2023 = glob.glob("../data/2023/*.csv")
file_list_2024 = glob.glob("../data/2024/*.csv")

train_df = read_and_concat(file_list_2022 + file_list_2023 + file_list_2024)
train_df = train_df.dropna(subset=["FinishingPos"])

print(f"Training samples: {len(train_df)} rows across {train_df['RacesInGEEra'].nunique()} races")

Training samples: 1357 rows across 68 races


## 3. Feature / Target Split

In [ ]:
X_train = train_df.drop(["FinishingPos", "Retired"], axis=1)
y_train = train_df["FinishingPos"]

print("Feature matrix:", X_train.shape)
print("Features:", list(X_train.columns))

## 4. Random Forest — Baseline Model

`RandomizedSearchCV` samples 50 configurations from a broad hyperparameter grid.  
Scoring: negative MSE with 3-fold cross-validation.

In [ ]:
rfr_param_dist = {
    "n_estimators":      list(range(1, 1001, 10)),
    "max_depth":         list(range(1, 101)),
    "min_samples_split": list(range(2, 101)),
    "min_samples_leaf":  list(range(1, 101)),
    "max_features":      ["sqrt", "log2", None],
    "bootstrap":         [True],
    "max_samples":       [0.5, 0.6, 0.7, 0.8, 0.9],
    "oob_score":         [True],
    "criterion":         ["absolute_error"],
}

rfr_model = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=30),
    param_distributions=rfr_param_dist,
    n_iter=50,
    cv=3,
    verbose=2,
    n_jobs=-1,
    scoring="neg_mean_squared_error",
    random_state=30,
)

rfr_model.fit(X_train, y_train)
print("Best RFR params:", rfr_model.best_params_)

In [ ]:
with open("models/rfr_model.pkl", "wb") as f:
    pickle.dump(rfr_model, f)

print("RFR model saved to models/rfr_model.pkl")

## 5. XGBoost — Final Model

Uses a `rank:ndcg` objective — the model learns to rank drivers within each race group  
rather than predicting raw finishing positions. `BayesSearchCV` is used for hyperparameter  
optimisation: it builds a probabilistic surrogate model over the search space, making it  
significantly more sample-efficient than random or grid search.

The XGBoost estimator is wrapped in a `Pipeline` to keep the tuning interface clean  
and to make it straightforward to add preprocessing steps later if needed.

In [ ]:
xgb_pipe = Pipeline(steps=[
    ("clf", xgb.XGBRegressor(random_state=30))
])

xgb_param_space = {
    "clf__objective":                    ["rank:ndcg"],
    "clf__eval_metric":                  ["ndcg"],
    "clf__tree_method":                  ["hist"],
    "clf__device":                       ["cuda"],
    "clf__grow_policy":                  ["lossguide"],
    "clf__max_leaves":                   Integer(32, 64),
    "clf__n_estimators":                 Integer(300, 1200),
    "clf__learning_rate":                Real(0.02, 0.17),
    "clf__subsample":                    Real(0.6, 0.85),
    "clf__colsample_bytree":             Real(0.6, 0.85),
    "clf__min_child_weight":             [1],
    "clf__gamma":                        Real(0, 1),
    "clf__lambda":                       Real(0.1, 1),
    "clf__alpha":                        Real(0, 0.75),
    "clf__lambdarank_num_pair_per_sample": Integer(2, 4),
    "clf__seed":                         [30],
}

xgb_model = BayesSearchCV(
    xgb_pipe,
    xgb_param_space,
    cv=5,
    n_iter=50,
    scoring="neg_mean_squared_error",
    random_state=30,
    verbose=2,
)

xgb_model.fit(X_train, y_train)
print("Best XGB params:", xgb_model.best_params_)

### Feature Importances

In [ ]:
importances = xgb_model.best_estimator_.named_steps["clf"].feature_importances_

importances_df = pd.DataFrame({
    "Feature":    X_train.columns,
    "Importance": importances,
}).sort_values(by="Importance", ascending=False)

print(importances_df.to_string(index=False))

In [ ]:
with open("models/xgb_model.pkl", "wb") as f:
    pickle.dump(xgb_model, f)

print("XGB model saved to models/xgb_model.pkl")